In [ ]:
!pip install statsmodels

## Ridge Regression

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge, RidgeCV, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import cross_val_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

In [27]:
df

,Transacted Price ($),Area (SQFT),Unit Price ($ PSF),Dist to CBD in Km,Sale Months Since Sep 2025,Num MRT Within 1km,Num Hawker Within 1km,Num Malls Within 1km,Num Hospitals Within 5km,Num Schools Within 2km,Num Parks Within 1km,Floor_Level_Category,Type_of_Sale_Encoded,Property_Type_Encoded,Market_Segment_Encoded,District
0,-0.211237,-0.460443,0.774350,1.347678,-1.117103,-1.177227,-1.175060,-1.296450,-1.143547,-1.118512,-1.374335,-1.619747,-1.228179,-0.334688,1.124971,1
1,-0.087630,-0.460443,1.240198,1.347678,-1.117103,-1.177227,-1.175060,-1.296450,-1.143547,-1.118512,-1.374335,0.947931,-1.228179,-0.334688,1.124971,1
2,-0.303362,-0.331436,-0.058154,-0.618881,-1.117103,-0.723616,-0.407922,0.640686,0.276567,-0.286674,0.161507,-0.152502,0.814214,2.987854,-0.888912,1
3,-0.100129,-0.442004,1.103809,1.347678,-1.117103,-1.177227,-1.175060,-1.296450,-1.143547,-1.118512,-1.374335,0.214309,-1.228179,-0.334688,1.124971,1
4,1.493910,2.267081,-0.950882,-0.363513,-1.117103,0.183604,-0.407922,0.156402,-0.670176,-0.286674,-0.030473,-1.619747,0.814214,-0.334688,-0.888912,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108367,0.046775,0.568074,-0.689954,-0.705343,1.573169,-0.762994,-0.374606,-1.216935,-1.442506,-1.156276,0.444833,1.136026,0.794933,0.657310,0.080720,28
108368,-0.016193,0.688792,-0.856036,-0.313104,1.573169,-0.287893,-0.374606,0.110836,0.451368,-0.384110,2.193072,-1.058519,0.794933,0.657310,0.080720,28
108369,-0.769843,1.443211,-1.977090,-0.350310,1.573169,-0.287893,-0.374606,-0.553049,0.451368,0.388056,-0.254463,-1.058519,0.794933,0.657310,0.080720,28
108370,0.656779,1.865669,-1.053869,-1.312109,1.573169,-1.238095,-0.374606,-1.216935,0.451368,-1.156276,-0.604110,-1.058519,0.794933,0.657310,-12.388450,28


## Ridge Regression

In [46]:
# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*80)
print("COMPREHENSIVE PROPERTY VALUE ANALYSIS WITH RIDGE REGRESSION")
print("="*80)

# ============================================================================
# STEP 1: LOAD AND PREPARE DATA
# ============================================================================

print("\n" + "="*80)
print("STEP 1: DATA PREPARATION")
print("="*80)

df = pd.read_csv('../datasets/normalized_data/combined_normalized.csv')

print(f"✅ Loaded {len(df)} property transactions")
print(f"   Date range: {df['Sale Months Since Sep 2025'].min():.2f} to {df['Sale Months Since Sep 2025'].max():.2f} months")
print(f"   Districts: {df['District'].nunique()}")
print(f"   Features: {len(df.columns)}")

# Check for missing values
missing = df.isnull().sum()
if missing.sum() > 0:
    print(f"\n⚠️ WARNING: Found {missing.sum()} missing values")
    print(missing[missing > 0])
    df = df.dropna()
    print(f"   After dropping: {len(df)} transactions remain")
else:
    print(f"\n✅ No missing values found")

# ============================================================================
# STEP 2: FEATURE ENGINEERING - CREATE DISTRICT DUMMIES
# ============================================================================

print("\n" + "="*80)
print("STEP 2: FEATURE ENGINEERING")
print("="*80)

# Define feature groups
location_features = ['Dist to CBD in Km']
amenity_features = [
    'Num MRT Within 1km', 
    'Num Hawker Within 1km', 
    'Num Malls Within 1km', 
    'Num Hospitals Within 5km', 
    'Num Schools Within 2km', 
    'Num Parks Within 1km'
]
property_features = [
    'Area (SQFT)',
    'Floor_Level_Category', 
    'Type_of_Sale_Encoded', 
    'Property_Type_Encoded', 
    'Market_Segment_Encoded'
]
time_features = ['Sale Months Since Sep 2025']

# Combine all base features
base_features = location_features + amenity_features + property_features + time_features

print(f"\n📊 Feature Summary:")
print(f"   Location features: {len(location_features)}")
print(f"   Amenity features: {len(amenity_features)}")
print(f"   Property features: {len(property_features)}")
print(f"   Time features: {len(time_features)}")

# Create district dummies (drop first to avoid multicollinearity)
district_dummies = pd.get_dummies(df['District'], prefix='District', drop_first=True)

print(f"   District dummies: {len(district_dummies.columns)} (baseline: District {df['District'].min()})")

# Prepare features and target
X_base = df[base_features].copy()
X = pd.concat([X_base, district_dummies], axis=1)
y = df['Transacted Price ($)'].copy()

print(f"   TOTAL FEATURES: {X.shape[1]}")
print(f"   TOTAL SAMPLES: {len(X)}")

# ============================================================================
# STEP 3: MULTICOLLINEARITY CHECK (VIF ANALYSIS)
# ============================================================================

print("\n" + "="*80)
print("STEP 3: MULTICOLLINEARITY CHECK (VIF)")
print("="*80)

# Calculate VIF for base features only (exclude district dummies)
vif_data = pd.DataFrame()
vif_data["Feature"] = base_features

vif_values = []
for i in range(len(base_features)):
    try:
        vif = variance_inflation_factor(X[base_features].values, i)
        vif_values.append(vif)
    except:
        vif_values.append(np.nan)

vif_data["VIF"] = vif_values
vif_data = vif_data.sort_values('VIF', ascending=False)

print(f"\n📊 Variance Inflation Factor (VIF) Analysis:")
print(f"   VIF < 5:  Low multicollinearity ✅")
print(f"   VIF 5-10: Moderate multicollinearity ⚠️")
print(f"   VIF > 10: High multicollinearity ❌")
print("\n" + "-"*60)
print(vif_data.to_string(index=False))

high_vif = vif_data[vif_data['VIF'] > 10]
if len(high_vif) > 0:
    print(f"\n⚠️ WARNING: {len(high_vif)} features have high multicollinearity")
    print(high_vif[['Feature', 'VIF']].to_string(index=False))
    print(f"\n   Ridge regression will help stabilize coefficients!")
else:
    print(f"\n✅ All features have acceptable VIF values")

# ============================================================================
# STEP 4: FIND OPTIMAL RIDGE ALPHA USING CROSS-VALIDATION
# ============================================================================

print("\n" + "="*80)
print("STEP 4: FINDING OPTIMAL RIDGE REGULARIZATION (ALPHA)")
print("="*80)

# Test wider range of alpha values
alphas = np.logspace(-3, 3, 100)  # From 0.001 to 1000

ridge_cv = RidgeCV(alphas=alphas, cv=10, scoring='r2')
ridge_cv.fit(X, y)

optimal_alpha = ridge_cv.alpha_
print(f"\n✅ Optimal Alpha: {optimal_alpha:.6f}")
print(f"   10-Fold CV R²: {ridge_cv.best_score_:.4f}")
print(f"\n📌 Using ALL {len(X)} samples for analysis (no train/test split)")
print(f"   Cross-validation ensures robust parameter selection")

# Visualize alpha selection
print(f"\n📊 Generating alpha selection plot...")
plt.figure(figsize=(12, 6))
cv_scores = []
for alpha in alphas:
    ridge_temp = Ridge(alpha=alpha)
    scores = cross_val_score(ridge_temp, X, y, cv=10, scoring='r2')
    cv_scores.append(scores.mean())

plt.plot(alphas, cv_scores, linewidth=2, color='steelblue')
plt.axvline(optimal_alpha, color='red', linestyle='--', linewidth=2, 
            label=f'Optimal α = {optimal_alpha:.4f}')
plt.xscale('log')
plt.xlabel('Alpha (Regularization Strength)', fontsize=12, fontweight='bold')
plt.ylabel('Cross-Validated R² Score', fontsize=12, fontweight='bold')
plt.title('Ridge Regression: Alpha Selection via 10-Fold Cross-Validation', 
          fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('./outputs/ridge/ridge_alpha_selection.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"   ✅ Saved: ./outputs/ridge/ridge_alpha_selection.png")

# ============================================================================
# STEP 5: FIT FINAL RIDGE MODEL ON ALL DATA
# ============================================================================

print("\n" + "="*80)
print("STEP 5: FINAL RIDGE REGRESSION MODEL")
print("="*80)

# Use optimal alpha from cross-validation
model = Ridge(alpha=optimal_alpha)
model.fit(X, y)

print(f"\n✅ Ridge Regression Model:")
print(f"   Alpha: {optimal_alpha:.6f}")
print(f"   Features: {len(model.coef_)} (all retained)")
print(f"   Samples: {len(X)}")

# ============================================================================
# STEP 6: MODEL PERFORMANCE EVALUATION
# ============================================================================

print("\n" + "="*80)
print("STEP 6: MODEL PERFORMANCE METRICS")
print("="*80)

# Predictions on all data
y_pred = model.predict(X)

# Calculate metrics
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
mae = mean_absolute_error(y, y_pred)

# Calculate MAPE (Mean Absolute Percentage Error)
def mape(y_true, y_pred):
    # Convert back to actual prices for MAPE calculation
    y_true_actual = y_true * df['Transacted Price ($)'].std() + df['Transacted Price ($)'].mean()
    y_pred_actual = y_pred * df['Transacted Price ($)'].std() + df['Transacted Price ($)'].mean()
    
    mask = y_true_actual != 0
    return np.mean(np.abs((y_true_actual[mask] - y_pred_actual[mask]) / y_true_actual[mask])) * 100


print(f"\n📊 MODEL FIT METRICS (All Data):")
print(f"   R² Score:  {r2:.4f}  ({r2*100:.2f}% of variance explained)")
print(f"   RMSE:      {rmse:.4f} (normalized)")
print(f"   MAE:       {mae:.4f} (normalized)")

# Cross-validation for robustness check
print(f"\n📊 10-Fold Cross-Validation (Robustness Check):")
cv_r2_scores = cross_val_score(model, X, y, cv=10, scoring='r2')
cv_mae_scores = -cross_val_score(model, X, y, cv=10, scoring='neg_mean_absolute_error')

print(f"   R² Score:")
print(f"      Mean: {cv_r2_scores.mean():.4f}")
print(f"      Std:  {cv_r2_scores.std():.4f}")
print(f"      Range: [{cv_r2_scores.min():.4f}, {cv_r2_scores.max():.4f}]")

print(f"\n   MAE:")
print(f"      Mean: {cv_mae_scores.mean():.4f}")
print(f"      Std:  {cv_mae_scores.std():.4f}")

# Interpret the gap
cv_gap = r2 - cv_r2_scores.mean()
print(f"\n📉 MODEL STABILITY:")
print(f"   R² Gap (Full Model - CV Mean): {cv_gap:.4f}")
if cv_gap > 0.05:
    print(f"   ⚠️ Moderate optimism - model fits training data better than CV suggests")
elif cv_gap > 0.02:
    print(f"   ✅ Good - small optimism is normal")
else:
    print(f"   ✅ Excellent - very stable model!")

print(f"\n📌 Interpretation:")
print(f"   • R² = {r2:.4f} means the model explains {r2*100:.1f}% of price variation")
print(f"   • Cross-validation R² = {cv_r2_scores.mean():.4f} suggests similar performance")
print(f"     on unseen data folds")

# Residual analysis
residuals = y - y_pred
print(f"\n📊 RESIDUAL ANALYSIS:")
print(f"   Mean residual: {residuals.mean():.6f} (should be ~0)")
print(f"   Std residual:  {residuals.std():.4f}")
print(f"   Min residual:  {residuals.min():.4f}")
print(f"   Max residual:  {residuals.max():.4f}")

COMPREHENSIVE PROPERTY VALUE ANALYSIS WITH RIDGE REGRESSION

STEP 1: DATA PREPARATION
✅ Loaded 108372 property transactions
   Date range: -4.39 to 2.88 months
   Districts: 27
   Features: 16

✅ No missing values found

STEP 2: FEATURE ENGINEERING

📊 Feature Summary:
   Location features: 1
   Amenity features: 6
   Property features: 5
   Time features: 1
   District dummies: 26 (baseline: District 1)
   TOTAL FEATURES: 39
   TOTAL SAMPLES: 108372

STEP 3: MULTICOLLINEARITY CHECK (VIF)

📊 Variance Inflation Factor (VIF) Analysis:
   VIF < 5:  Low multicollinearity ✅
   VIF 5-10: Moderate multicollinearity ⚠️
   VIF > 10: High multicollinearity ❌

------------------------------------------------------------
                   Feature      VIF
         Dist to CBD in Km 1.800276
  Num Hospitals Within 5km 1.550484
      Num Malls Within 1km 1.286866
        Num MRT Within 1km 1.242021
    Market_Segment_Encoded 1.232555
    Num Schools Within 2km 1.194910
      Type_of_Sale_Encoded 1.1

In [47]:
# ============================================================================
# STEP 7: COEFFICIENT ANALYSIS - UNDERSTANDING FEATURE RELATIONSHIPS
# ============================================================================

print("\n" + "="*80)
print("STEP 7: COEFFICIENT ANALYSIS")
print("="*80)

# Extract all coefficients
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', ascending=False)

# Separate district effects from other features
district_coefs = coef_df[coef_df['Feature'].str.startswith('District_')].copy()
other_coefs = coef_df[~coef_df['Feature'].str.startswith('District_')].copy()

# ============================================================================
# A. BASE FEATURE EFFECTS
# ============================================================================

print("\n" + "-"*80)
print("A. BASE FEATURE COEFFICIENTS (Non-District)")
print("-"*80)
print(other_coefs.to_string(index=False))

print("\n💡 INTERPRETATION:")
print("   • Positive coefficient → Increases property price")
print("   • Negative coefficient → Decreases property price")
print("   • Magnitude indicates strength of relationship")
print("   • All features are normalized (mean=0, std=1) for fair comparison")

# Highlight top positive and negative effects
print("\n📊 TOP POSITIVE EFFECTS:")
top_positive = other_coefs.nlargest(3, 'Coefficient')
for idx, row in top_positive.iterrows():
    print(f"   • {row['Feature']}: {row['Coefficient']:.4f}")
    
print("\n📊 TOP NEGATIVE EFFECTS:")
top_negative = other_coefs.nsmallest(3, 'Coefficient')
for idx, row in top_negative.iterrows():
    print(f"   • {row['Feature']}: {row['Coefficient']:.4f}")

# ============================================================================
# B. DISTRICT EFFECTS
# ============================================================================

print("\n" + "-"*80)
print("B. DISTRICT COEFFICIENTS (vs. Baseline District)")
print("-"*80)

baseline_district = df['District'].min()
print(f"\n📍 Baseline District: District {baseline_district}")
print(f"   All other districts are compared to this baseline\n")

print(district_coefs.to_string(index=False))

print("\n💡 INTERPRETATION:")
print(f"   • Positive coefficient → More expensive than District {baseline_district}")
print(f"   • Negative coefficient → Less expensive than District {baseline_district}")
print("   • Coefficient shows the price premium/discount")

# Highlight premium and budget districts
print("\n🏆 PREMIUM DISTRICTS (Highest Price Premium):")
premium_districts = district_coefs.nlargest(5, 'Coefficient')
for idx, row in premium_districts.iterrows():
    district_num = row['Feature'].replace('District_', '')
    print(f"   • District {district_num}: +{row['Coefficient']:.4f} (vs. baseline)")

print("\n💰 BUDGET DISTRICTS (Largest Price Discount):")
budget_districts = district_coefs.nsmallest(5, 'Coefficient')
for idx, row in budget_districts.iterrows():
    district_num = row['Feature'].replace('District_', '')
    print(f"   • District {district_num}: {row['Coefficient']:.4f} (vs. baseline)")

# ============================================================================
# C. COEFFICIENT VISUALIZATIONS
# ============================================================================

print("\n" + "-"*80)
print("C. GENERATING COEFFICIENT VISUALIZATIONS")
print("-"*80)

# 1. Base Features Coefficient Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Sort and color code
other_coefs_sorted = other_coefs.sort_values('Coefficient')
colors = ['red' if x < 0 else 'green' for x in other_coefs_sorted['Coefficient']]

ax1.barh(range(len(other_coefs_sorted)), other_coefs_sorted['Coefficient'], color=colors, alpha=0.7)
ax1.set_yticks(range(len(other_coefs_sorted)))
ax1.set_yticklabels(other_coefs_sorted['Feature'], fontsize=9)
ax1.set_xlabel('Coefficient Value', fontsize=12, fontweight='bold')
ax1.set_title('Base Feature Coefficients\n(Green = Positive Effect, Red = Negative Effect)', 
              fontsize=13, fontweight='bold')
ax1.axvline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
ax1.grid(axis='x', alpha=0.3)

# 2. District Coefficients Plot
district_coefs_sorted = district_coefs.sort_values('Coefficient')
colors_district = ['red' if x < 0 else 'green' for x in district_coefs_sorted['Coefficient']]
district_labels = [f"D{x.replace('District_', '')}" for x in district_coefs_sorted['Feature']]

ax2.barh(range(len(district_coefs_sorted)), district_coefs_sorted['Coefficient'], 
         color=colors_district, alpha=0.7)
ax2.set_yticks(range(len(district_coefs_sorted)))
ax2.set_yticklabels(district_labels, fontsize=8)
ax2.set_xlabel('Coefficient Value (vs. Baseline)', fontsize=12, fontweight='bold')
ax2.set_title(f'District Price Effects (vs. District {baseline_district})\n(Green = Premium, Red = Discount)', 
              fontsize=13, fontweight='bold')
ax2.axvline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('./outputs/ridge/coefficient_analysis.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"   ✅ Saved: ./outputs/ridge/coefficient_analysis.png")

# 3. Feature Importance by Category
print("\n📊 Generating feature importance by category...")

feature_categories = {
    'Location': location_features,
    'Amenities': amenity_features,
    'Property': property_features,
    'Time': time_features
}

category_importance = {}
for category, features in feature_categories.items():
    # Sum of absolute coefficients in category
    coefs_in_category = other_coefs[other_coefs['Feature'].isin(features)]['Coefficient'].abs().sum()
    category_importance[category] = coefs_in_category

category_df = pd.DataFrame(list(category_importance.items()), 
                          columns=['Category', 'Total Absolute Coefficient'])
category_df = category_df.sort_values('Total Absolute Coefficient', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(category_df['Category'], category_df['Total Absolute Coefficient'], 
              color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'], alpha=0.7)
ax.set_ylabel('Total Absolute Coefficient', fontsize=12, fontweight='bold')
ax.set_xlabel('Feature Category', fontsize=12, fontweight='bold')
ax.set_title('Feature Category Importance\n(Sum of Absolute Coefficients)', 
             fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}',
            ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('./outputs/ridge/category_importance.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"   ✅ Saved: ./outputs/ridge/category_importance.png")

print("\n" + category_df.to_string(index=False))

# ============================================================================
# D. COEFFICIENT SUMMARY TABLE
# ============================================================================

print("\n" + "-"*80)
print("D. COEFFICIENT SUMMARY")
print("-"*80)

summary_stats = pd.DataFrame({
    'Category': ['Base Features', 'District Effects', 'All Features'],
    'Count': [len(other_coefs), len(district_coefs), len(coef_df)],
    'Positive': [
        (other_coefs['Coefficient'] > 0).sum(),
        (district_coefs['Coefficient'] > 0).sum(),
        (coef_df['Coefficient'] > 0).sum()
    ],
    'Negative': [
        (other_coefs['Coefficient'] < 0).sum(),
        (district_coefs['Coefficient'] < 0).sum(),
        (coef_df['Coefficient'] < 0).sum()
    ],
    'Mean Abs Coef': [
        other_coefs['Coefficient'].abs().mean(),
        district_coefs['Coefficient'].abs().mean(),
        coef_df['Coefficient'].abs().mean()
    ],
    'Max Positive': [
        other_coefs['Coefficient'].max(),
        district_coefs['Coefficient'].max(),
        coef_df['Coefficient'].max()
    ],
    'Max Negative': [
        other_coefs['Coefficient'].min(),
        district_coefs['Coefficient'].min(),
        coef_df['Coefficient'].min()
    ]
})

print("\n" + summary_stats.to_string(index=False))

print("\n" + "="*80)
print("✅ COEFFICIENT ANALYSIS COMPLETE")
print("="*80)


STEP 7: COEFFICIENT ANALYSIS

--------------------------------------------------------------------------------
A. BASE FEATURE COEFFICIENTS (Non-District)
--------------------------------------------------------------------------------
                   Feature  Coefficient
               Area (SQFT)     0.836657
      Floor_Level_Category     0.056694
      Num Parks Within 1km     0.036412
  Num Hospitals Within 5km     0.018162
      Num Malls Within 1km     0.015113
    Num Schools Within 2km    -0.005891
     Property_Type_Encoded    -0.010539
     Num Hawker Within 1km    -0.010891
    Market_Segment_Encoded    -0.012146
        Num MRT Within 1km    -0.029893
         Dist to CBD in Km    -0.043978
Sale Months Since Sep 2025    -0.240600
      Type_of_Sale_Encoded    -0.345832

💡 INTERPRETATION:
   • Positive coefficient → Increases property price
   • Negative coefficient → Decreases property price
   • Magnitude indicates strength of relationship
   • All features are normal

## Ridge Regression Per District

In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import cross_val_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*80)
print("RIDGE REGRESSION ANALYSIS - PER DISTRICT")
print("="*80)

# ============================================================================
# STEP 1: LOAD DATA
# ============================================================================

print("\n" + "="*80)
print("STEP 1: DATA LOADING")
print("="*80)

# Load the FULL dataset with all districts
df_full = pd.read_csv('../datasets/normalized_data/combined_normalized.csv')

print(f"✅ Loaded {len(df_full)} total property transactions")
print(f"   Total features: {len(df_full.columns)}")

# Check for missing values
missing = df_full.isnull().sum()
if missing.sum() > 0:
    print(f"\n⚠️ WARNING: Found {missing.sum()} missing values")
    df_full = df_full.dropna()
    print(f"   After dropping: {len(df_full)} transactions remain")
else:
    print(f"\n✅ No missing values found")

# Extract postal district from the data
# Assuming you have a 'Postal District' column or can extract it
if 'Postal District' in df_full.columns:
    postal_district_col = 'Postal District'
elif 'District' in df_full.columns:
    postal_district_col = 'District'
else:
    # If not available, create from address or other info
    print("⚠️ No district column found - creating from data...")
    # You may need to adjust this based on your data structure

# Get unique districts
districts = sorted(df_full[postal_district_col].unique())
print(f"\n📍 Found {len(districts)} unique postal districts:")
print(f"   Districts: {districts}")

# ============================================================================
# STEP 2: DEFINE FEATURES
# ============================================================================

print("\n" + "="*80)
print("STEP 2: FEATURE DEFINITION")
print("="*80)

# Define feature groups (EXCLUDING district dummies - we're doing per-district analysis)
location_features = ['Dist to CBD in Km']
amenity_features = [
    'Num MRT Within 1km', 
    'Num Hawker Within 1km', 
    'Num Malls Within 1km', 
    'Num Hospitals Within 5km', 
    'Num Schools Within 2km', 
    'Num Parks Within 1km'
]
property_features = [
    'Area (SQFT)',
    'Floor_Level_Category', 
    'Type_of_Sale_Encoded', 
    'Property_Type_Encoded', 
    'Market_Segment_Encoded'
]
time_features = ['Sale Months Since Sep 2025']

# Combine all features
all_features = location_features + amenity_features + property_features + time_features

print(f"\n📊 Feature Summary:")
print(f"   Location features: {len(location_features)}")
print(f"   Amenity features: {len(amenity_features)}")
print(f"   Property features: {len(property_features)}")
print(f"   Time features: {len(time_features)}")
print(f"   TOTAL FEATURES: {len(all_features)}")

# ============================================================================
# STEP 3: PER-DISTRICT RIDGE REGRESSION
# ============================================================================

print("\n" + "="*80)
print("STEP 3: RIDGE REGRESSION - DISTRICT BY DISTRICT")
print("="*80)

# Storage for results
district_results = []
district_models = {}
district_coefficients = {}

# Analyze each district
for district in districts:
    print(f"\n{'='*80}")
    print(f"ANALYZING DISTRICT {district}")
    print(f"{'='*80}")
    
    # Filter data for this district
    df_district = df_full[df_full[postal_district_col] == district].copy()
    
    print(f"\n📊 District {district} Statistics:")
    print(f"   Total transactions: {len(df_district)}")
    print(f"   Price range: ${df_district['Transacted Price ($)'].min():,.0f} - ${df_district['Transacted Price ($)'].max():,.0f}")
    print(f"   Median price: ${df_district['Transacted Price ($)'].median():,.0f}")
    
    # Check if we have enough samples
    if len(df_district) < 50:
        print(f"   ⚠️ WARNING: Only {len(df_district)} samples - skipping (need at least 50)")
        continue
    
    # Prepare features and target
    X_district = df_district[all_features].copy()
    y_district = df_district['Transacted Price ($)'].copy()
    
    print(f"\n🔍 Feature Matrix:")
    print(f"   Shape: {X_district.shape}")
    print(f"   Target shape: {y_district.shape}")
    
    # Find optimal alpha using cross-validation
    print(f"\n🎯 Finding optimal alpha...")
    alphas = np.logspace(-3, 3, 100)
    
    try:
        ridge_cv = RidgeCV(alphas=alphas, cv=min(10, len(df_district)//10), scoring='r2')
        ridge_cv.fit(X_district, y_district)
        
        optimal_alpha = ridge_cv.alpha_
        cv_r2 = ridge_cv.best_score_
        
        print(f"   ✅ Optimal alpha: {optimal_alpha:.6f}")
        print(f"   ✅ CV R² score: {cv_r2:.4f}")
        
        # Fit final model with optimal alpha
        model = Ridge(alpha=optimal_alpha)
        model.fit(X_district, y_district)
        
        # Make predictions
        y_pred = model.predict(X_district)
        
        # Calculate metrics
        r2 = r2_score(y_district, y_pred)
        rmse = np.sqrt(mean_squared_error(y_district, y_pred))
        mae = mean_absolute_error(y_district, y_pred)
        
        # Calculate MAPE
        mape_value = np.mean(np.abs((y_district - y_pred) / y_district)) * 100
        
        print(f"\n📊 Model Performance:")
        print(f"   R² Score:  {r2:.4f}")
        print(f"   RMSE:      ${rmse:,.0f}")
        print(f"   MAE:       ${mae:,.0f}")
        print(f"   MAPE:      {mape_value:.2f}%")
        
        # Store results
        district_results.append({
            'District': district,
            'Num_Samples': len(df_district),
            'Optimal_Alpha': optimal_alpha,
            'CV_R2': cv_r2,
            'R2_Score': r2,
            'RMSE': rmse,
            'MAE': mae,
            'MAPE': mape_value,
            'Median_Price': df_district['Transacted Price ($)'].median()
        })
        
        # Store model and coefficients
        district_models[district] = model
        
        coef_dict = dict(zip(all_features, model.coef_))
        coef_dict['Intercept'] = model.intercept_
        district_coefficients[district] = coef_dict
        
    except Exception as e:
        print(f"   ❌ Error fitting model: {str(e)}")
        continue

# ============================================================================
# STEP 4: COMPARE RESULTS ACROSS DISTRICTS
# ============================================================================

print("\n" + "="*80)
print("STEP 4: CROSS-DISTRICT COMPARISON")
print("="*80)

# Create results DataFrame
results_df = pd.DataFrame(district_results)
results_df = results_df.sort_values('R2_Score', ascending=False)

print("\n📊 MODEL PERFORMANCE SUMMARY (Ranked by R²):")
print(results_df.to_string(index=False))

# Save results
results_df.to_csv('./outputs/ridge/district_comparison.csv', index=False)
print(f"\n✅ Saved: ./outputs/ridge/district_comparison.csv")

# ============================================================================
# STEP 5: VISUALIZATIONS
# ============================================================================

print("\n" + "="*80)
print("STEP 5: GENERATING VISUALIZATIONS")
print("="*80)

# 1. R² Comparison across districts
plt.figure(figsize=(14, 6))
bars = plt.bar(results_df['District'].astype(str), results_df['R2_Score'], 
               color='steelblue', edgecolor='black', linewidth=1.5)

# Color code by performance
for i, bar in enumerate(bars):
    r2 = results_df.iloc[i]['R2_Score']
    if r2 >= 0.8:
        bar.set_color('green')
    elif r2 >= 0.6:
        bar.set_color('orange')
    else:
        bar.set_color('red')

plt.axhline(y=results_df['R2_Score'].mean(), color='red', linestyle='--', 
            linewidth=2, label=f'Mean R² = {results_df["R2_Score"].mean():.3f}')
plt.xlabel('Postal District', fontsize=12, fontweight='bold')
plt.ylabel('R² Score', fontsize=12, fontweight='bold')
plt.title('Ridge Regression Performance by District', fontsize=14, fontweight='bold')
plt.ylim(0, 1)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('./outputs/ridge/district_r2_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"   ✅ Saved: ./outputs/ridge/district_r2_comparison.png")

# 2. Sample size vs R²
plt.figure(figsize=(10, 6))
plt.scatter(results_df['Num_Samples'], results_df['R2_Score'], 
            s=200, alpha=0.6, c=results_df['R2_Score'], cmap='RdYlGn', 
            edgecolor='black', linewidth=1.5)

for idx, row in results_df.iterrows():
    plt.annotate(f"D{row['District']}", 
                 (row['Num_Samples'], row['R2_Score']),
                 fontsize=9, fontweight='bold')

plt.colorbar(label='R² Score')
plt.xlabel('Number of Samples', fontsize=12, fontweight='bold')
plt.ylabel('R² Score', fontsize=12, fontweight='bold')
plt.title('Model Performance vs Sample Size', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('./outputs/ridge/sample_size_vs_r2.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"   ✅ Saved: ./outputs/ridge/sample_size_vs_r2.png")

# 3. Coefficient heatmap across districts
coef_df = pd.DataFrame(district_coefficients).T
coef_df = coef_df[all_features]  # Exclude intercept for clarity

plt.figure(figsize=(14, max(8, len(districts)*0.5)))
sns.heatmap(coef_df, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            cbar_kws={'label': 'Coefficient Value'}, linewidths=0.5)
plt.xlabel('Features', fontsize=12, fontweight='bold')
plt.ylabel('District', fontsize=12, fontweight='bold')
plt.title('Ridge Regression Coefficients Across Districts', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./outputs/ridge/coefficient_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"   ✅ Saved: ./outputs/ridge/coefficient_heatmap.png")

# 4. Feature importance comparison (top 5 features per district)
print(f"\n📊 Top 5 Most Important Features by District:")
for district in districts:
    if district in district_coefficients:
        coefs = district_coefficients[district]
        # Remove intercept
        coefs_features = {k: v for k, v in coefs.items() if k != 'Intercept'}
        # Sort by absolute value
        top_features = sorted(coefs_features.items(), key=lambda x: abs(x[1]), reverse=True)[:5]
        
        print(f"\n   District {district}:")
        for feature, coef in top_features:
            print(f"      {feature:30s}: {coef:+.4f}")

print("\n" + "="*80)
print("✅ DISTRICT-LEVEL RIDGE REGRESSION ANALYSIS COMPLETE!")
print("="*80)

RIDGE REGRESSION ANALYSIS - PER DISTRICT

STEP 1: DATA LOADING
✅ Loaded 108372 total property transactions
   Total features: 16

✅ No missing values found

📍 Found 27 unique postal districts:
   Districts: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(25), np.int64(26), np.int64(27), np.int64(28)]

STEP 2: FEATURE DEFINITION

📊 Feature Summary:
   Location features: 1
   Amenity features: 6
   Property features: 5
   Time features: 1
   TOTAL FEATURES: 13

STEP 3: RIDGE REGRESSION - DISTRICT BY DISTRICT

ANALYZING DISTRICT 1

📊 District 1 Statistics:
   Total transactions: 1638
   Price range: $-1 - $13
   Median price: $-0

🔍 Feature Matrix:
   Shape: (1638, 13)
   Target shape: (1638,)

🎯 Finding optimal alpha.